# PRAKTIKUM 3 — Manipulasi Data Tabular dengan Pandas
**Mata kuliah:** Pengantar Machine Learning · **Minggu:** 4  
**Nama:** [Mochammad Maril Livio] · **NIM:** [231230065]

## 1. Membuat DataFrame, memilih data, dan melakukan filtering

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 28],
    'Salary': [50000, 60000, 70000, 55000],
    'Department': ['IT', 'HR', 'IT', 'Finance']
})
print('DataFrame:\n', df)
print('Kolom nama:', df['Name'].tolist())
print('Baris indeks 1:\n', df.iloc[1])
print('Karyawan IT:\n', df.loc[df['Department'] == 'IT', ['Name', 'Salary']])

DataFrame:
       Name  Age  Salary Department
0    Alice   25   50000         IT
1      Bob   30   60000         HR
2  Charlie   35   70000         IT
3    David   28   55000    Finance
Kolom nama: ['Alice', 'Bob', 'Charlie', 'David']
Baris indeks 1:
 Name            Bob
Age              30
Salary        60000
Department       HR
Name: 1, dtype: object
Karyawan IT:
       Name  Salary
0    Alice   50000
2  Charlie   70000


## 2. Manipulasi kolom, missing values, grouping, aggregation dan sorting

In [2]:
df['Senior'] = df['Age'] >= 35
print('Kolom Senior:\n', df[['Name', 'Senior']])

contoh_nan = df.copy()
contoh_nan.loc[1, 'Salary'] = np.nan
print('NaN per kolom:\n', contoh_nan.isna().sum())
contoh_nan['Salary'] = contoh_nan['Salary'].fillna(contoh_nan['Salary'].median())
print('Setelah isi NaN:\n', contoh_nan)

Kolom Senior:
       Name  Senior
0    Alice   False
1      Bob   False
2  Charlie    True
3    David   False
NaN per kolom:
 Name          0
Age           0
Salary        1
Department    0
Senior        0
dtype: int64
Setelah isi NaN:
       Name  Age   Salary Department  Senior
0    Alice   25  50000.0         IT   False
1      Bob   30  55000.0         HR   False
2  Charlie   35  70000.0         IT    True
3    David   28  55000.0    Finance   False


In [3]:
rata_gaji = df.groupby('Department')['Salary'].mean().sort_values(ascending=False)
print('Rata-rata gaji per departemen:\n', rata_gaji)
print('Statistik deskriptif:\n', df[['Age', 'Salary']].describe())
print('Diurutkan berdasarkan gaji:\n', df.sort_values('Salary', ascending=False)[['Name', 'Salary']])

data_dept = pd.DataFrame({'Department': ['IT', 'HR', 'Finance'], 'Location': ['Jakarta', 'Bandung', 'Pontianak']})
print('Hasil merge:\n', df.merge(data_dept, on='Department', how='left')[['Name','Department','Location']])

Rata-rata gaji per departemen:
 Department
HR         60000.0
IT         60000.0
Finance    55000.0
Name: Salary, dtype: float64
Statistik deskriptif:
              Age        Salary
count   4.000000      4.000000
mean   29.500000  58750.000000
std     4.203173   8539.125638
min    25.000000  50000.000000
25%    27.250000  53750.000000
50%    29.000000  57500.000000
75%    31.250000  62500.000000
max    35.000000  70000.000000
Diurutkan berdasarkan gaji:
       Name  Salary
2  Charlie   70000
1      Bob   60000
3    David   55000
0    Alice   50000
Hasil merge:
       Name Department   Location
0    Alice         IT    Jakarta
1      Bob         HR    Bandung
2  Charlie         IT    Jakarta
3    David    Finance  Pontianak


## 3. Tanggal dan operasi teks

In [4]:
df['Join_Date'] = pd.date_range('2023-01-01', periods=len(df))
df['Year'] = df['Join_Date'].dt.year
df['Name_Upper'] = df['Name'].str.upper()
df['Name_Length'] = df['Name'].str.len()
print(df[['Name', 'Join_Date', 'Year', 'Name_Upper', 'Name_Length']])

      Name  Join_Date  Year Name_Upper  Name_Length
0    Alice 2023-01-01  2023      ALICE            5
1      Bob 2023-01-02  2023        BOB            3
2  Charlie 2023-01-03  2023    CHARLIE            7
3    David 2023-01-04  2023      DAVID            5


## 4. Latihan 3 
**TODO 1:** total revenue per produk. **TODO 2:** pelanggan dengan total belanja tertinggi. **TODO 3:** revenue harian. **TODO 4:** deteksi anomali jumlah barang memakai Z-score.

In [5]:
# Dataset latihan dosen: 1.000 transaksi retail, Januari–Maret 2023
np.random.seed(42)
dates = pd.date_range('2023-01-01', '2023-03-31', freq='D')
retail_data = pd.DataFrame({
    'date': np.random.choice(dates, 1000),
    'product_id': np.random.randint(1, 11, 1000),
    'quantity': np.random.randint(1, 10, 1000),
    'price': np.random.uniform(10, 100, 1000),
    'customer_id': np.random.randint(1, 101, 1000)
})
retail_data['revenue'] = retail_data['quantity'] * retail_data['price']
print('Contoh transaksi:\n', retail_data.head())

Contoh transaksi:
         date  product_id  quantity      price  customer_id     revenue
0 2023-02-21           9         4  44.581566           84  178.326263
1 2023-01-15           5         5  50.511103           76  252.555513
2 2023-03-13           3         4  62.133972           33  248.535887
3 2023-03-02           5         3  78.341592           28  235.024777
4 2023-01-21           5         8  18.898626            8  151.189005


In [6]:
# TODO 1 — pendapatan total per produk
revenue_per_product = retail_data.groupby('product_id')['revenue'].sum().sort_values(ascending=False)

# TODO 2 — lima pelanggan dengan total belanja terbesar
top_customers = retail_data.groupby('customer_id')['revenue'].sum().nlargest(5)

# TODO 3 — deret waktu pendapatan per hari
daily_revenue = retail_data.groupby('date')['revenue'].sum().sort_index()

# TODO 4 — anomali kuantitas dengan z-score
def detect_quantity_anomalies(df, threshold=2):
    q = df['quantity'].astype(float)
    rata, simpangan = q.mean(), q.std(ddof=0)
    if simpangan == 0:
        return df.iloc[0:0].copy()
    z = (q - rata) / simpangan
    return df.loc[z.abs() > threshold].copy()

anomalies = detect_quantity_anomalies(retail_data)
print('Pendapatan per produk:\n', revenue_per_product)
print('Top 5 pelanggan:\n', top_customers)
print('Pendapatan harian (5 pertama):\n', daily_revenue.head())
print('Jumlah transaksi anomali:', len(anomalies))

Pendapatan per produk:
 product_id
8     32062.915807
9     30812.566464
7     30786.212454
5     29894.628309
6     29771.241361
2     28433.373576
3     27360.489903
10    25146.428705
1     24953.204554
4     23950.525430
Name: revenue, dtype: float64
Top 5 pelanggan:
 customer_id
38    5239.536260
59    5185.221233
89    4759.149270
94    4690.528278
15    4657.289888
Name: revenue, dtype: float64
Pendapatan harian (5 pertama):
 date
2023-01-01    5268.121555
2023-01-02    4656.102661
2023-01-03    3226.578585
2023-01-04    2462.071219
2023-01-05    3741.645829
Name: revenue, dtype: float64
Jumlah transaksi anomali: 0


In [7]:
assert len(revenue_per_product) <= 10
assert len(top_customers) == 5
assert abs(revenue_per_product.sum()-retail_data['revenue'].sum()) < 1e-6
assert abs(daily_revenue.sum()-retail_data['revenue'].sum()) < 1e-6
assert len(daily_revenue) <= len(dates)
contoh_outlier = pd.DataFrame({'quantity':[1,1,1,1,1,1,1,100]})
assert len(detect_quantity_anomalies(contoh_outlier, threshold=2)) == 1
print('✅ Praktikum 3 dan semua latihan berhasil.')

✅ Praktikum 3 dan semua latihan berhasil.
